# CH101 production mesh intake

이 Notebook은 실제 고해상도 생산용 CH101_A_HighRes_Production_v001.blend를 받은 뒤 실행한다.
자동화된 primitive/styled blockout을 생산 모델로 승격하지 않으며, 이 검사는 Gate B의 기술 입력 검증만 수행한다.
시각 품질과 최종 승인은 사람이 별도로 확인해야 한다.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
from google.colab import files

TOOLS_REPO_URL = 'https://github.com/siri2677/re-camp-blender.git'
TOOLS_COMMIT = 'c2f8247ec4fd9b29877ff38b92af64eca18f56aa'
ART_COMMIT = 'b6c9b3128358e061eee6184230929413eba84101'
TOOLS_DIR = Path('/content/re-camp-blender')
OUTPUT_DIR = Path('/content/re-camp-production-intake')
EXPECTED_BLEND_NAME = 'CH101_A_HighRes_Production_v001.blend'
DEFAULT_BLEND = Path('/content') / EXPECTED_BLEND_NAME
PRODUCTION_BLEND = Path(os.environ.get('RECAMP_CH101_PRODUCTION_BLEND', str(DEFAULT_BLEND)))
REPORT_PATH = OUTPUT_DIR / 'ch101-mesh-intake-report.json'
HANDOFF_PATH = OUTPUT_DIR / 'production-mesh-handoff.json'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if not TOOLS_DIR.exists():
    subprocess.run(['git', 'clone', '--no-checkout', TOOLS_REPO_URL, str(TOOLS_DIR)], check=True)
subprocess.run(['git', '-C', str(TOOLS_DIR), 'fetch', '--depth', '1', 'origin', TOOLS_COMMIT], check=True)
subprocess.run(['git', '-C', str(TOOLS_DIR), 'checkout', '--detach', TOOLS_COMMIT], check=True)
print(f'Tools: {TOOLS_DIR}')
print(f'Input: {PRODUCTION_BLEND}')

In [ ]:
def command_exists(name):
    return shutil.which(name) is not None

if not command_exists('blender') or not command_exists('xvfb-run'):
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'blender', 'xvfb'], check=True)

print(subprocess.run(['blender', '--version'], check=True, capture_output=True, text=True).stdout.splitlines()[0])
print('xvfb-run:', shutil.which('xvfb-run'))

In [ ]:
if not PRODUCTION_BLEND.is_file():
    print('Upload exactly one high-resolution production .blend file.')
    uploaded = files.upload()
    blend_names = [name for name in uploaded if name.lower().endswith('.blend')]
    if len(blend_names) != 1:
        raise ValueError(f'Expected exactly one .blend upload, got: {blend_names}')
    if blend_names[0] != EXPECTED_BLEND_NAME:
        raise ValueError(f'Expected {EXPECTED_BLEND_NAME}, got: {blend_names[0]}')
    PRODUCTION_BLEND = Path('/content') / blend_names[0]

if PRODUCTION_BLEND.suffix.lower() != '.blend' or not PRODUCTION_BLEND.is_file():
    raise FileNotFoundError(f'Production .blend not found: {PRODUCTION_BLEND}')
if PRODUCTION_BLEND.name != EXPECTED_BLEND_NAME:
    raise ValueError(f'Expected {EXPECTED_BLEND_NAME}, got: {PRODUCTION_BLEND.name}')
SOURCE_COPY = OUTPUT_DIR / PRODUCTION_BLEND.name
if PRODUCTION_BLEND.resolve() != SOURCE_COPY.resolve():
    shutil.copy2(PRODUCTION_BLEND, SOURCE_COPY)
PRODUCTION_BLEND = SOURCE_COPY
print(f'Using production mesh: {PRODUCTION_BLEND}')

In [ ]:
validator = TOOLS_DIR / 'scripts' / 'blender' / 'validate_ch101_mesh_intake.py'
command = [
    'xvfb-run', '-a', 'blender', '-b', '--python', str(validator), '--',
    '--blend', str(PRODUCTION_BLEND), '--report', str(REPORT_PATH),
]
result = subprocess.run(command, text=True, capture_output=True)
print(result.stdout)
print(result.stderr)
if result.returncode != 0 or not REPORT_PATH.is_file():
    raise RuntimeError('CH101 production mesh intake command failed; inspect Blender output above.')

report = json.loads(REPORT_PATH.read_text(encoding='utf-8'))
if report.get('status') != 'PASS':
    raise RuntimeError(f'Production mesh intake did not pass: {report.get("status")}')

sha256 = hashlib.sha256(PRODUCTION_BLEND.read_bytes()).hexdigest()
handoff = {
    'character': 'CH101',
    'sourceStatus': 'PRODUCTION_MESH_READY',
    'gateB': 'PENDING_HUMAN_REVIEW',
    'unityInputAllowed': False,
    'sourceReference': report.get('source_reference', ''),
    'contractVersion': 'current-roster-socket-contract-v001',
    'artCommit': ART_COMMIT,
    'toolsCommit': TOOLS_COMMIT,
    'blend': PRODUCTION_BLEND.name,
    'blendSha256': sha256,
    'validator': 'validate_current_roster_mesh_intake.py',
    'validatorReport': REPORT_PATH.name,
}
HANDOFF_PATH.write_text(json.dumps(handoff, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(json.dumps(handoff, ensure_ascii=False, indent=2))

In [ ]:
archive = Path('/content/CH101-production-mesh-intake.zip')
if archive.exists():
    archive.unlink()
shutil.make_archive(str(archive.with_suffix('')), 'zip', OUTPUT_DIR)
archive_sha256 = hashlib.sha256(archive.read_bytes()).hexdigest()
checksum = archive.with_name(archive.name + '.sha256')
checksum.write_text(f'{archive_sha256}  {archive.name}\n', encoding='utf-8')
print(f'Created: {archive}')
print(f'Created: {checksum}')
files.download(str(archive))
files.download(str(checksum))